In [1]:
import os
import time

import polars as pl

In [ ]:
# MAGIA (não mexer mt pfv)
def show_args(func):
    def wrapper(*args, **kwargs):
        arg_list = []

        # Prepare argument list
        for name, value in zip(func.__code__.co_varnames, args):
            if isinstance(value, pl.DataFrame):
                arg_list.append(f"{name}: DataFrame ({value.shape})")
            else:
                arg_list.append(f"{name}: {value}")
        for k, v in kwargs.items():
            if isinstance(v, pl.DataFrame):
                arg_list.append(f"{k}: DataFrame ({v.shape})")
            else:
                arg_list.append(f"{k}: {v}")

        #print(f"Executando <{func.__name__}>\n({'\\n'.join(arg_list)})")

        # Measure execution time
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()

        print(f"Tempo de execução: {end - start:.6f} segundos\n")
        return result
    return wrapper

In [ ]:
from pathlib import Path

@show_args
def csvs_to_csv(input_folder, output_folder):
    input_path = Path(input_folder)
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    for csv_file in input_path.glob("*.csv"):
        try:
            df = pl.read_csv(csv_file, encoding="utf-8")
        except Exception:
            print(f"{csv_file} could not be read with UTF-8, trying Latin1...")
            df = pl.read_csv(csv_file, encoding="latin1")

        csv_file = output_path / csv_file.with_suffix(".csv").name
        df.write_csv(csv_file)

In [ ]:
csvs_to_csv("../data/data/refined/csvs", "../data/data/refined/csvs")

In [2]:
articles_raw = pl.read_csv('../data/data/refined/csvs/article.csv').rename({
    'eid': 'art_id',
    'subtype description': 'art_subtype',
    'title': 'art_title',
    'published date': 'art_published_date',
    'cited by count': 'art_citations',
    'source id' : 'art_source',
    'aggregation type' : 'art_source_type'
}).select(['art_id', 'art_subtype', 'art_title', 'art_published_date', 'art_citations', 'art_source', 'art_source_type'])

scores_raw = pl.read_csv('../data/data/refined/csvs/citescore.csv', null_values=['#N/A']).rename({
    'Scopus Source ID' : 'art_source',
    'Title' : 'src_title',
    'Citation Count' : 'src_citation_count',
    'CiteScore' : 'src_citescore',
    'Scopus Sub-Subject Area' : 'src_subject_area',
    'Percentile' : 'src_percentile',
    'RANK' : 'src_rank',
    'Rank Out Of' : 'src_rank_count',
    'Quartile' : 'src_quartile'
}).select(['art_source', 'src_title', 'src_citation_count', 'src_citescore', 'src_subject_area', 'src_percentile', 'src_rank', 'src_rank_count', 'src_quartile'])

authors_institutions_raw = pl.read_csv('../data/data/refined/csvs/authors_institution.csv').rename({
    'eid': 'art_id',
    'auid': 'aut_id',
    'creator': 'main_writer',
    'afid': 'ins_id',
    'dptid': 'dpt_id',
    'organization': 'ins_org',
    'country': 'ins_country',
    'city': 'ins_city'
}).with_columns(
    pl.col('ins_id').cast(pl.Utf8)
)

institutions_raw = (pl.read_csv('../data/data/refined/csvs/institutions.csv').rename({
        'Código Mantenedora': 'ins_maint_id',
        'Categoria': 'ins_category',
        'afid': 'ins_id',
        'Situação da IES': 'ins_status',
        'Código IES': 'ins_code',
        'Instituição(IES)': 'ins_name'
    }).select(['ins_maint_id', 'ins_category', 'ins_id', 'ins_status', 'ins_code', 'ins_name'])
).with_columns(
    pl.col('ins_id').cast(pl.Utf8)
)

authors_raw = pl.read_csv('../data/data/refined/csvs/author.csv').rename({
    'auid': 'aut_id',
    'given name': 'aut_name',
    'surname': 'aut_surname',
    'indexed name': 'aut_indexed_name'
}).select(['aut_id', 'aut_name', 'aut_surname', 'aut_indexed_name'])

subject_areas_raw = pl.read_csv('../data/data/refined/csvs/subject_area.csv')
subject_areas_raw = subject_areas_raw.rename({
    subject_areas_raw.columns[0]: 'sub_id',
    subject_areas_raw.columns[1]: 'src_subject_area',
    subject_areas_raw.columns[2]: 'src_subject_macro'
})

In [3]:
main = (
    articles_raw
    .join(scores_raw, on='art_source', how='inner')
    .join(authors_institutions_raw, on='art_id', how='inner')
    .join(institutions_raw, on='ins_id', how='inner')
    .join(authors_raw, on='aut_id', how='inner')
    .join(subject_areas_raw, on='src_subject_area', how='inner')
)

In [ ]:
institutions_raw.select('ins_id').value_counts()

ins_id
126381129.0    2
129321446.0    2
124439778.0    2
120096371.0    2
115111107.0    2
              ..
130585251.0    1
131393270.0    1
131651280.0    1
130784293.0    1
131850669.0    1
Name: count, Length: 10133, dtype: int64

In [ ]:
main.head()

,art_id,art_subtype,art_title,art_published_date,art_citations,art_source,art_source_type,src_title,src_citation_count,src_citescore,...,ins_maint_id,ins_category,ins_status,ins_code,ins_name,aut_name,aut_surname,aut_indexed_name,sub_id,src_subject_macro
0,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,UNIVERSIDADE DE SÃO PAULO (USP),Giliane,Belarmino,Belarmino G.,2701,MEDI
1,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,UNIVERSIDADE DE SÃO PAULO (USP),Lilian Mika,Horie,Horie L.M.,2701,MEDI
2,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,UNIVERSIDADE DE SÃO PAULO (USP),Priscila Campos,Sala,Sala P.C.,2701,MEDI
3,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,UNIVERSIDADE DE SÃO PAULO (USP),Raquel S.,Torrinhas,Torrinhas R.S.,2701,MEDI
4,2-s2.0-84951935294,Article,Body adiposity index performance in estimating...,2015-12-30,18.0,12653.0,Journal,Nutrition Journal,2502,6.7,...,15715,Publica,Ativa,55,UNIVERSIDADE DE SÃO PAULO (USP),Dan L.,Waitzberg,Waitzberg D.L.,2701,MEDI


In [ ]:
main.columns

Index(['art_id', 'art_subtype', 'art_title', 'art_published_date',
       'art_citations', 'art_source', 'art_source_type', 'src_title',
       'src_citation_count', 'src_citescore', 'src_subject_area',
       'src_percentile', 'src_rank', 'src_rank_count', 'src_quartile',
       'aut_id', 'main_writer', 'ins_id', 'dpt_id', 'ins_org', 'ins_country',
       'ins_city', 'ins_maint_id', 'ins_category', 'ins_status', 'ins_code',
       'ins_name', 'aut_name', 'aut_surname', 'aut_indexed_name', 'sub_id',
       'src_subject_macro'],
      dtype='str')

In [ ]:
ranking = main.select(['art_id', 'art_subtype', 'art_published_date', 'src_subject_area', 'src_subject_macro', 'src_quartile', 'ins_maint_id', 'ins_name', 'ins_category', 'ins_country', 'aut_id', 'src_percentile', 'ins_code'])
institution = main.select(['ins_maint_id', 'ins_name', 'ins_category', 'ins_status', 'ins_code']).unique(subset=['ins_maint_id'])

In [ ]:
print(f"Unique art_id in ranking: {ranking.select('art_id').n_unique()}")
print(f"Unique aut_id in ranking: {ranking.select('aut_id').n_unique()}")
print(f"Unique ins_maint_id in institution: {institution.select('ins_maint_id').n_unique()}")

Unique art_id in ranking: 652996
Unique aut_id in ranking: 631317
Unique ins_maint_id in institution: 1196


In [ ]:
def get_score_by_quartile(quartile: int, ranking: pl.DataFrame) -> pl.DataFrame:
    """
    Calculate interdisciplinarity (DI) values for articles filtered by quartile.
    
    This function filters articles and computes a DI_value based on the number of authors
    and optionally the quartile of the publication source. The DI_value represents the
    contribution of each article to institutional interdisciplinarity, normalized by
    the number of authors.
    
    Parameters:
    -----------
    quartile : int
        Quartile filter for source publications:
        - -1: No quartiles used in the calculation
        - 0: No quartile filter (all articles counted with base DI calculation)
        - 1, 2, 3, 4: Only articles from sources in specified quartile
    
    ranking : pl.DataFrame
        DataFrame to be filtered
    
    Returns:
    --------
    pl.DataFrame
        Modified ranking DataFrame with added 'DI_value' column containing:
        - quartile == -2: DI_value = 10 / (number_of_authors * number_of_areas * all_percentile)
        - quartile == -1: DI_value = 10 / (number_of_authors * number_of_areas)
        - quartile == 0: DI_value = 10 / (number_of_authors * number_of_areas * all_quartile)
        - quartile in [1,2,3,4]: DI_value = 10 / (number_of_authors * number_of_areas * specified_quartile)
    
    Notes:
    ------
    - Only articles (art_subtype == 'Article') are included
    - DI_value is normalized by dividing by the number of authors per article
    - Higher quartile values (top journals) result in lower DI_value when quartile != 0
    - This weighting accounts for institutional credit distributed across multi-author papers
    """
    ranking_article = ranking.filter(pl.col('art_subtype') == 'Article').clone()
    
    # Calculate author counts per article
    author_counts = ranking_article.group_by('art_id').agg(pl.col('aut_id').count().alias('author_count'))
    ranking_article = ranking_article.join(author_counts, on='art_id', how='left')
    
    if quartile == -2:
        ranking_article = ranking_article.with_columns(
            (pl.col('src_percentile') / pl.col('author_count')).alias('DI_value')
        ).drop('src_quartile')
    elif quartile == -1:
        ranking_article = ranking_article.with_columns(
            (10 / pl.col('author_count')).alias('DI_value')
        ).drop('src_percentile')
    elif quartile == 0:
        ranking_article = ranking_article.with_columns(
            (10 / (pl.col('author_count') * pl.col('src_quartile'))).alias('DI_value')
        ).drop('src_percentile')
    else:
        ranking_article = ranking_article.filter(pl.col('src_quartile') == quartile)
        ranking_article = ranking_article.with_columns(
            (10 / (pl.col('author_count') * pl.col('src_quartile'))).alias('DI_value')
        ).drop('src_percentile')
    
    return ranking_article.drop('author_count')

In [ ]:
def calculate_di_tables(ranking_article: pl.DataFrame, institution: pl.DataFrame):
    """
    Calculate DI (Interdisciplinarity) tables at both subject and macro levels.
    
    Parameters:
    -----------
    ranking_article : pl.DataFrame
        DataFrame containing article data with DI_value column
    institution : pl.DataFrame
        DataFrame containing institution information with ins_maint_id
    
    Returns:
    --------
    tuple of (di_table_subs, di_table_macros)
        - di_table_subs: DI table at subject area level with only ins_name and subject columns
        - di_table_macros: DI table at macro area level with only ins_name and macro columns
    """
    # Subject area level
    sub_sums = ranking_article.group_by(['ins_code', 'src_subject_area']).agg(
        pl.col('DI_value').sum()
    ).pivot(index='ins_code', columns='src_subject_area', values='DI_value', aggregate_function='sum')

    di_table_subs = institution.join(
        sub_sums, left_on='ins_code', right_on='ins_code', how='left'
    ).fill_null(0)
    
    di_table_subs = di_table_subs.drop(['ins_code', 'ins_category', 'ins_status'])

    # Macro area level
    macro_sums = ranking_article.group_by(['ins_code', 'src_subject_macro']).agg(
        pl.col('DI_value').sum()
    ).pivot(index='ins_code', columns='src_subject_macro', values='DI_value', aggregate_function='sum')

    di_table_macros = institution.join(
        macro_sums, left_on='ins_code', right_on='ins_code', how='left'
    ).fill_null(0)
    
    di_table_macros = di_table_macros.drop(['ins_maint_id', 'ins_category', 'ins_status', 'ins_code'])

    return di_table_subs, di_table_macros

In [ ]:
def order_by_non_zero_count(df: pl.DataFrame) -> pl.DataFrame:
    """
    Orders DataFrame rows by the number of non-zero values in ascending order (fewest zeros first).
    
    Parameters:
    -----------
    df : pl.DataFrame
        Input dataframe to be sorted
    
    Returns:
    --------
    pl.DataFrame
        DataFrame sorted by non-zero count (rows with more non-zero values come first)
    """
    # Calculate non-zero counts for each row
    non_zero_counts = df.select([
        pl.when(pl.col(col) != 0).then(1).otherwise(0).sum().over(pl.lit(1)).alias('non_zero_count')
        for col in df.columns if col != 'ins_name'
    ]).select(pl.sum_horizontal('*').alias('non_zero_count'))
    
    # Add non-zero count to dataframe and sort
    df_with_counts = df.with_columns(pl.lit(0).alias('_temp')).with_columns(
        pl.when((pl.col('_temp') != 0) | (pl.col(df.columns[1]) != 0))
        .then(1).otherwise(0).sum().over(pl.all()).alias('non_zero_count')
    )
    
    # Simpler approach: count non-zero values per row using a custom approach
    non_zero_list = []
    for row in df.iter_rows(named=True):
        count = sum(1 for col in df.columns if col != 'ins_name' and row[col] != 0)
        non_zero_list.append(count)
    
    df_sorted = df.with_columns(pl.Series('_non_zero', non_zero_list)).sort('_non_zero', descending=True).drop('_non_zero')
    return df_sorted

In [ ]:
def get_output_paths(quartile: int) -> tuple:
    """
    Generate output directory paths based on quartile value.
    
    Parameters:
    -----------
    quartile : int
        Quartile filter value:
        - -1: Returns paths for 'no_quality' directories
        - 0: Returns paths for 'quartile' directories
        - 1, 2, 3, 4: Returns paths for 'q{i}' directories
    
    Returns:
    --------
    tuple of (macro_path, sub_path)
        Paths for saving macro and subject level ranking files
    """
    macro_path = './data/macro/macro_'
    sub_path = './data/sub/sub_'
    if quartile == -2:
        macro_path += 'percentile'
        sub_path += 'percentile'
    elif quartile == -1:
        macro_path += 'no_quality'
        sub_path += 'no_quality'
    elif quartile == 0:
        macro_path += 'quartile'
        sub_path += 'quartile'
    else:
        macro_path += f'q{quartile}'
        sub_path += f'q{quartile}'
    
    return macro_path, sub_path


def save_ranking_tables_with_path(di_table_subs: pl.DataFrame, di_table_macros: pl.DataFrame, 
                                   macro_path: str, sub_path: str, top_n: int = 200):
    """
    Save ranking tables at macro and subject levels to specified paths.
    
    Parameters:
    -----------
    di_table_subs : pl.DataFrame
        DI table at subject area level with ins_name and subject area columns
    di_table_macros : pl.DataFrame
        DI table at macro area level with ins_name and macro area columns
    macro_path : str
        Base directory path for macro-level output files
    sub_path : str
        Base directory path for subject-level output files
    top_n : int, optional
        Number of top institutions to retain (default: 200)
    
    Returns:
    --------
    None
        Files are saved to disk in the specified directories
    """
    
    # Create directories if they don't exist
    os.makedirs(macro_path, exist_ok=True)
    os.makedirs(sub_path, exist_ok=True)
    
    # Process macro-level rankings
    final_leys_macros = order_by_non_zero_count(di_table_macros)[:top_n]
    final_leys_macros.select('ins_name').write_csv(f'{macro_path}/institutions_names.csv', include_header=False)
    
    labels_macros = final_leys_macros.columns[1:]  # Skip 'ins_name' column
    with open(f'{macro_path}/labels.csv', 'w') as f:
        f.write('\n'.join(labels_macros))
    
    final_leys_macros_drop = final_leys_macros.drop('ins_name')
    final_leys_macros_drop.write_csv(f'{macro_path}/rank.txt', include_header=False)

    # Process subject-level rankings
    final_leys_subs = order_by_non_zero_count(di_table_subs)[:top_n]
    final_leys_subs.select('ins_name').write_csv(f'{sub_path}/institutions_names.csv', include_header=False)
    
    labels_subs = final_leys_subs.columns[1:]  # Skip 'ins_name' column
    with open(f'{sub_path}/labels.csv', 'w') as f:
        f.write('\n'.join(labels_subs))
    
    final_leys_subs_drop = final_leys_subs.drop('ins_name')
    final_leys_subs_drop.write_csv(f'{sub_path}/rank.txt', include_header=False)

In [ ]:
# Execute for all quartile values
for quartile in range(-2, 5):
    print(f"\nProcessing quartile: {quartile}")

    # Calculate DI values for this quartile
    ranking_article = get_score_by_quartile(quartile, ranking)
    print(ranking_article.head())
    
    # Calculate DI tables
    di_table_subs, di_table_macros = calculate_di_tables(ranking_article, institution)
    
    # Get output paths
    macro_path, sub_path = get_output_paths(quartile)
    
    # Save ranking tables to appropriate paths
    save_ranking_tables_with_path(di_table_subs, di_table_macros, macro_path, sub_path, top_n=200)
    
    print(f"\tSaved to: {macro_path} and {sub_path}")

print("\nAll quartiles processed successfully!")


Processing quartile: -2
               art_id art_subtype art_published_date  \
0  2-s2.0-84951935294     Article         2015-12-30   
1  2-s2.0-84951935294     Article         2015-12-30   
2  2-s2.0-84951935294     Article         2015-12-30   
3  2-s2.0-84951935294     Article         2015-12-30   
4  2-s2.0-84951935294     Article         2015-12-30   

           src_subject_area src_subject_macro  ins_maint_id  \
0  Medicine (miscellaneous)              MEDI         15715   
1  Medicine (miscellaneous)              MEDI         15715   
2  Medicine (miscellaneous)              MEDI         15715   
3  Medicine (miscellaneous)              MEDI         15715   
4  Medicine (miscellaneous)              MEDI         15715   

                          ins_name ins_category ins_country       aut_id  \
0  UNIVERSIDADE DE SÃO PAULO (USP)      Publica      Brazil  56862304200   
1  UNIVERSIDADE DE SÃO PAULO (USP)      Publica      Brazil  16241649700   
2  UNIVERSIDADE DE SÃO PAULO (U